# Citations

Citations let Claude ground each claim in the **exact span of your source document** — turning it from a black box into a research assistant that shows its work. Users can verify answers against the original text.

**Enable it** by adding two fields to a `document` block:

```python
{
    "type": "document",
    "source": { ... },
    "title": "earth.pdf",
    "citations": {"enabled": True},
}
```

**Each citation carries:**
- `cited_text` — the exact supporting text from your document
- `document_index` / `document_title` — which document (useful with several)
- **PDF source** → `start_page_number` / `end_page_number`
- **Plain-text source** → `start_char_index` / `end_char_index`

> Uses `claude-haiku-4-5-20251001`. Reuses `earth.pdf` from the PDF lesson; also shows a short plain-text source for char-index citations.

## Setup

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import os
import base64
from anthropic import Anthropic
from anthropic.types import Message

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return message


SECTION = os.path.join(os.path.dirname(find_dotenv()), "01-building-with-the-claude-api", "06-features")
print("Ready.")

Ready.


## Rendering citations

With citations on, the response is a series of `text` blocks, each with an attached `citations` list. This helper prints each snippet of the answer followed by the source span(s) that back it — page range for PDFs, char range for text.

In [2]:
def _location(c):
    if getattr(c, "start_page_number", None) is not None:
        return f"pages {c.start_page_number}-{c.end_page_number}"
    if getattr(c, "start_char_index", None) is not None:
        return f"chars {c.start_char_index}-{c.end_char_index}"
    return "?"


def render_with_citations(message):
    for block in message.content:
        if block.type != "text":
            continue
        print(block.text)
        for c in (getattr(block, "citations", None) or []):
            title = getattr(c, "document_title", "") or ""
            cited = (getattr(c, "cited_text", "") or "").strip().replace("\n", " ")
            if len(cited) > 140:
                cited = cited[:140] + "..."
            print(f"   \u21b3 [{title} | {_location(c)}] \"{cited}\"")

## Citations from a PDF

`title` + `citations: {enabled: True}` on the document block. Ask a question and Claude answers with page-level citations into `earth.pdf`.

In [3]:
with open(os.path.join(SECTION, "earth.pdf"), "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(messages, [
    {
        "type": "document",
        "source": {"type": "base64", "media_type": "application/pdf", "data": file_bytes},
        "title": "earth.pdf",
        "citations": {"enabled": True},
    },
    {"type": "text", "text": "How were Earth's atmosphere and oceans formed?"},
])

response = chat(messages)
render_with_citations(response)

Earth's atmosphere and oceans were formed by volcanic activity and outgassing. Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.
 these sources condensed into th..."formed by volcanic activity and outgassing.


## Citations from plain text

Citations aren't PDF-only. Use a `text/plain` source with `citations` enabled and you get **character positions** instead of page numbers — precise spans into the raw string.

In [4]:
article_text = (
    "Earth's atmosphere and oceans were formed by volcanic activity and outgassing. "
    "Water vapor from these sources condensed into the oceans, augmented by water and "
    "ice from asteroids, protoplanets, and comets. By 3.5 billion years ago, Earth's "
    "magnetic field was established, which helped prevent the atmosphere from being "
    "stripped away by the solar wind."
)

messages = []
add_user_message(messages, [
    {
        "type": "document",
        "source": {"type": "text", "media_type": "text/plain", "data": article_text},
        "title": "earth_article",
        "citations": {"enabled": True},
    },
    {"type": "text", "text": "How were Earth's atmosphere and oceans formed?"},
])

response = chat(messages)
render_with_citations(response)

Earth's atmosphere and oceans were formed by volcanic activity and outgassing.
   ↳ [earth_article | chars 0-79] "Earth's atmosphere and oceans were formed by volcanic activity and outgassing."
 
Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.
   ↳ [earth_article | chars 79-206] "Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets."
 Additionally, 
by 3.5 billion years ago, Earth's magnetic field was established, which helped prevent the atmosphere from being stripped away by the solar wind.
   ↳ [earth_article | chars 206-351] "By 3.5 billion years ago, Earth's magnetic field was established, which helped prevent the atmosphere from being stripped away by the solar ..."


Inspect one citation object to see the exact fields (char indices for the text source):

In [5]:
for block in response.content:
    if block.type == "text" and getattr(block, "citations", None):
        print(block.citations[0])
        break

CitationCharLocation(cited_text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing. ", document_index=0, document_title='earth_article', end_char_index=79, file_id=None, start_char_index=0, type='char_location')


## When to use citations

- Users need to **verify** information for accuracy
- You're working with **authoritative** documents users should be able to reference
- **Transparency** about sources is critical
- Users may want to explore the **surrounding context** of a fact

The real payoff is in the UI: hover a claim to reveal its source span, letting users trust and drill into the material. This pairs naturally with the **RAG** section — cite the exact retrieved chunk a claim came from.

Next: **Prompt caching** — reuse a large, stable context (like a big document) across requests to cut cost and latency.